In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    MultiLabelBinarizer,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

import joblib

In [19]:
def squeeze_column(x):
    return x.squeeze()

In [20]:
df = pd.read_csv("generated_data.csv").drop_duplicates()

df.columns = df.columns.str.strip().str.upper()
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (253, 29)


,AGE,GENDER,DEPARTMENT,CHIEF_COMPLAINTS,COMORBIDITIES,RISKFACTORS,SURGICAL_HISTORY,SOCIAL_HISTORY,DIAGNOSIS,CLASSIFICATION_OF_UTI,...,WBC,POLYMORPHS,CRP,RFT_SERUM_CREATININE,SERUM_URIC_ACID,BLOOD_UREA,CUE_PUS_CELLS,EPITHELIAL_CELLS,PROTEINS,RBC
0,34,Female,General Medicine,Dysuria;Increased frequency,NaN,Poor hygiene,NaN,Non smoker,Acute cystitis,Uncomplicated,...,9800,68,6,0.9,4.8,28,12,4,Trace,3.0
1,52,Male,Urology,Fever;Flank pain;Dysuria,Diabetes,Catheterization,NaN,Non smoker,Acute pyelonephritis,Complicated,...,18200,78,65,2.1,6.2,48,28,6,Positive,6.0
2,27,Female,General Medicine,Dysuria;Urgency,NaN,Sexual activity,NaN,Non smoker,Acute cystitis,Uncomplicated,...,8700,60,4,0.8,4.1,24,10,5,Negative,2.0
3,68,Male,ICU,Fever;Flank pain;Vomiting,Diabetes;Hypertension,Catheterization,Prostate surgery,Ex smoker,Severe pyelonephritis,Complicated,...,22400,82,98,3.0,7.1,66,35,8,Positive,10.0
4,45,Female,General Medicine,Dysuria;Lower abdominal pain,Diabetes,NaN,Hysterectomy,Non smoker,Acute cystitis,Complicated,...,11200,70,18,1.1,5.0,32,18,6,Trace,4.0


In [21]:
numeric_features = [
    'AGE', 'CBP_LYMPHOCYTES', 'WBC', 'POLYMORPHS',
    'CRP', 'RFT_SERUM_CREATININE', 'SERUM_URIC_ACID',
    'BLOOD_UREA', 'CUE_PUS_CELLS', 'EPITHELIAL_CELLS', 'RBC'
]

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [22]:
df['TYPE_OF_BACTERIA'] = df['TYPE_OF_BACTERIA'].str.strip().str.lower()

df['TYPE_OF_BACTERIA_ENC'] = df['TYPE_OF_BACTERIA'].map({
    'gram negative': 0,
    'gram positive': 1
})

In [23]:
df['PREVIOUS_ANTIBIOTIC_USED'] = df['PREVIOUS_ANTIBIOTIC_USED'].fillna('')

df['PREV_ABX_LIST'] = df['PREVIOUS_ANTIBIOTIC_USED'].apply(
    lambda x: [i.strip() for i in x.split(';') if i.strip() != '']
)

prev_abx_mlb = MultiLabelBinarizer()

prev_abx_encoded = prev_abx_mlb.fit_transform(df['PREV_ABX_LIST'])

prev_abx_df = pd.DataFrame(
    prev_abx_encoded,
    columns=[f"PREV_ABX_{c}" for c in prev_abx_mlb.classes_],
    index=df.index
)

df = pd.concat([df, prev_abx_df], axis=1)

print("Previous Antibiotics Detected:")
print(prev_abx_mlb.classes_)

Previous Antibiotics Detected:
['Amoxicillin' 'Cefepime' 'Cefixime' 'Ceftriaxone' 'Ciprofloxacin'
 'Imipenem' 'Levofloxacin' 'Meropenem' 'Piperacillin-Tazobactam']


In [24]:
df['SENSITIVE'] = df['SENSITIVE'].fillna('')

df['SENSITIVE_LIST'] = df['SENSITIVE'].apply(
    lambda x: [i.strip() for i in x.split(';') if i.strip() != '']
)

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['SENSITIVE_LIST'])

print("Sensitive Antibiotics Detected:")
print(mlb.classes_)

Sensitive Antibiotics Detected:
['Amikacin' 'Cefepime' 'Cefixime' 'Ceftriaxone' 'Colistin' 'Fosfomycin'
 'Gentamicin' 'Linezolid' 'Meropenem' 'Nitrofurantoin'
 'Piperacillin-Tazobactam' 'Tigecycline' 'Vancomycin']


In [25]:
categorical_features = [
    'GENDER', 'DEPARTMENT',
    'CLASSIFICATION_OF_UTI', 'TYPE_OF_UTI',
    'SITE_OF_INFECTION', 'TYPE_OF_SAMPLE',
    'PROTEINS'
]

text_features = [
    'CHIEF_COMPLAINTS', 'COMORBIDITIES',
    'RISKFACTORS', 'SURGICAL_HISTORY',
    'SOCIAL_HISTORY', 'DIAGNOSIS'
]

prev_abx_feature_cols = prev_abx_df.columns.tolist()

feature_columns = (
    numeric_features +
    categorical_features +
    text_features +
    ['TYPE_OF_BACTERIA_ENC'] +
    prev_abx_feature_cols
)

X = df[feature_columns]

In [26]:
X.columns

Index(['AGE', 'CBP_LYMPHOCYTES', 'WBC', 'POLYMORPHS', 'CRP',
       'RFT_SERUM_CREATININE', 'SERUM_URIC_ACID', 'BLOOD_UREA',
       'CUE_PUS_CELLS', 'EPITHELIAL_CELLS', 'RBC', 'GENDER', 'DEPARTMENT',
       'CLASSIFICATION_OF_UTI', 'TYPE_OF_UTI', 'SITE_OF_INFECTION',
       'TYPE_OF_SAMPLE', 'PROTEINS', 'CHIEF_COMPLAINTS', 'COMORBIDITIES',
       'RISKFACTORS', 'SURGICAL_HISTORY', 'SOCIAL_HISTORY', 'DIAGNOSIS',
       'TYPE_OF_BACTERIA_ENC', 'PREV_ABX_Amoxicillin', 'PREV_ABX_Cefepime',
       'PREV_ABX_Cefixime', 'PREV_ABX_Ceftriaxone', 'PREV_ABX_Ciprofloxacin',
       'PREV_ABX_Imipenem', 'PREV_ABX_Levofloxacin', 'PREV_ABX_Meropenem',
       'PREV_ABX_Piperacillin-Tazobactam'],
      dtype='object')

In [27]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [28]:
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [29]:
text_transformers = []

for col in text_features:

    text_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='')),
        ('to_string', FunctionTransformer(
            squeeze_column, validate=False
        )),
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(1,2),
            min_df=2,
            max_df=0.9
        ))
    ])

    text_transformers.append((col, text_pipeline, [col]))

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
        ('bacteria', 'passthrough', ['TYPE_OF_BACTERIA_ENC']),
        ('prev_abx', 'passthrough', prev_abx_feature_cols),
        *text_transformers
    ],
    remainder='drop'
)

In [31]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', MultiOutputClassifier(
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    ))
])

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

In [33]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['AGE', 'CBP_LYMPHOCYTES',
                                                   'WBC', 'POLYMORPHS', 'CRP',
                                                   'RFT_SERUM_CREATININE',
                                                   'SERUM_URIC_ACID',
                                                   'BLOOD_UREA',
                                                   'CUE_PUS_CELLS',
                                                   'EPITHELIAL_CELLS', 'RBC']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleI...
                                                                   SimpleImputer(fill_value='',
                                                                                 strategy='constant')),
                                                                  ('to_string',
                                                                   FunctionTransformer(func=<function squeeze_column at 0x0000026158613BA0>)),
                                                                  ('tfidf',
                                                                   TfidfVectorizer(max_df=0.9,
                                                                                   min_df=2,
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   stop_words='english'))]),
                                                  ['DIAGNOSIS'])])),
                ('classifier',
                 MultiOutputClassifier(estimator=RandomForestClassifier(n_estimators=300,
                                                                        n_jobs=-1,
                                                                        random_state=42)))])

In [34]:
y_pred = model_pipeline.predict(X_test)

print("Classification Report (Per Antibiotic):")
print(classification_report(y_test, y_pred, target_names=mlb.classes_))

Classification Report (Per Antibiotic):
                         precision    recall  f1-score   support

               Amikacin       0.83      0.83      0.83        12
               Cefepime       0.00      0.00      0.00         0
               Cefixime       1.00      0.83      0.91         6
            Ceftriaxone       1.00      0.83      0.91        12
               Colistin       1.00      1.00      1.00        16
             Fosfomycin       0.88      1.00      0.93        14
             Gentamicin       0.00      0.00      0.00         0
              Linezolid       0.89      0.89      0.89         9
              Meropenem       1.00      0.33      0.50         3
         Nitrofurantoin       0.96      1.00      0.98        23
Piperacillin-Tazobactam       0.76      1.00      0.87        13
            Tigecycline       0.67      0.67      0.67         6
             Vancomycin       1.00      0.91      0.95        11

              micro avg       0.90      0.91    

c:\Users\bagam\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\bagam\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\bagam\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [35]:
model_pipeline.score(X_test, y_test)
print("Test Accuracy:", model_pipeline.score(X_test, y_test))

Test Accuracy: 0.7254901960784313


In [39]:
joblib.dump(model_pipeline, "model_3_sensitivity.pkl")
joblib.dump(mlb, "model_3_sensitive_mlb.pkl")
joblib.dump(prev_abx_mlb, "model_3_prev_abx_mlb.pkl")

print("Model 3 saved successfully.")

Model 3 saved successfully.
